# Named Entity Recognition

<b>Named Entity Recognition (NER)</b>: It is used to find and label important information (people, places, organizations, dates, etc.) in text.

• The default LLM for NER is <b>BERT</b> (encoder-only`)

NER is useful for information extraction, search engines, document processing, question answering, customer-support analysis, knowledge graphs, and extracting structured information from unstructured text.

In [2]:
import pandas as pd
import warnings
import numpy 

pd.set_option('max_colwidth',None)
warnings.filterwarnings('ignore')

df=pd.read_excel(r'C:\Users\HP\Desktop\PG\NLP\NLP with ML\data\Popchip_Reviews.xlsx')
df.head()

,Id,UserId,Rating,Priority,Title,Text
0,23689,A21SYVGVNG8RAS,5,Low,Yummy snacks!,Popchips are the bomb!! I use the parmesan garlic to scoop up cottage cheese as a healthy alternative to chips and dip. My healthy eating program is saved.
1,23690,AQJYXC0MPRQJL,5,Low,Great chip that is different from the rest,"I like the puffed nature of this chip that makes it more unique in the chip market. I ordered the Salt and Vinegar and absolutely love that flavor, hands down my favorite chip ever. I have tried the cheddar and regular flavors as well. The cheddar is about a 4/5 and the regular is about a 3/5 because I prefer strong flavors and obviously that would not be the case for the regular. The Salt and Vinegar is kind of weak compared to some regular S&V chips, but is quite flavorful and makes you wanting to come back for more."
2,23691,A30NYUHEDLWI0Y,5,Low,Great Alternative to Potato Chips,"I just love these chips! I was always a big fan of potato chips, but haven't had one since I discovered popchips. They are great for dipping or all alone. I am constantly re-ordering them. One note however-if you are on a low salt diet these chips are probably not for you. They are high in sodium. We go through a case every two months. If you love them it pays to join the subscribe and save program through Amazon. You save money and stay supplied!"
3,23692,A2NU55U9LKTB5J,3,High,Not somthing I would crave,"These tasted like potatoe stix, that we got in grade school with our lunches usually on pizza day. They were the bomb then, not so much now. Won't buy again unless I get them for cheap or free."
4,23693,A225F7QFP5LIW2,5,Low,healthy and delicious,"These chips are great! They look almost like a flattened rice cake, but taste so much better, more like a potato chip. The bbq flavor is delicious. They are very low in fat and full of flavor. It is easy to eat an entire bag of these!"


## Using Default Model

In [4]:
%%time

from transformers import pipeline

ner_analyzer=pipeline('ner',
                      model='dbmdz/bert-large-cased-finetuned-conll03-english',
                      device=-1,
                      aggregation_strategy='SIMPLE'
                     )

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 1756.26it/s]
[transformers] BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CPU times: total: 688 ms
Wall time: 2.62 s


In [8]:
ner_analyzer('Elon Musk founded SpaceX in California in 2002.')

[{'entity_group': 'PER',
  'score': np.float32(0.99861574),
  'word': 'Elon Musk',
  'start': 0,
  'end': 9},
 {'entity_group': 'ORG',
  'score': np.float32(0.9985991),
  'word': 'SpaceX',
  'start': 18,
  'end': 24},
 {'entity_group': 'LOC',
  'score': np.float32(0.99963474),
  'word': 'California',
  'start': 28,
  'end': 38}]

1. entity_group : This tells you what type of entity the model detected.<br>
PER → Person , ORG → Organization and LOC → Location<br>
<br>
2. score : This represents the model's confidence in its prediction.<br>
For example: 'score': 0.99861574 means the model is approximately 99.86% confident that "Elon Musk" is a person.<br>
<br>  
3. start and end : These indicate the character positions of the entity in the original sentence.

## Using Different Model (From The Model Hub)

In [9]:
ner_analyzer2=pipeline('ner',
                      model='dslim/bert-base-NER',
                      device=-1,
                      aggregation_strategy='SIMPLE'
                     )
ner_analyzer2('Elon Musk founded SpaceX in California in 2002.')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2519.78it/s]
[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'entity_group': 'ORG',
  'score': np.float32(0.6674712),
  'word': 'Elon',
  'start': 0,
  'end': 4},
 {'entity_group': 'PER',
  'score': np.float32(0.8312903),
  'word': 'Mu',
  'start': 5,
  'end': 7},
 {'entity_group': 'ORG',
  'score': np.float32(0.5448268),
  'word': '##sk',
  'start': 7,
  'end': 9},
 {'entity_group': 'ORG',
  'score': np.float32(0.9992529),
  'word': 'SpaceX',
  'start': 18,
  'end': 24},
 {'entity_group': 'LOC',
  'score': np.float32(0.9996731),
  'word': 'California',
  'start': 28,
  'end': 38}]

## Cleaned Output

In [11]:
# Example

entities=ner_analyzer2('Elon Musk founded SpaceX in California in 2002.')
[entity['word'] for entity in entities]

['Elon', 'Mu', '##sk', 'SpaceX', 'California']

In [12]:
df['Named Entities']=df['Text'].apply(lambda x: [entity['word'] for entity in ner_analyzer2(x)])

df.head()

,Id,UserId,Rating,Priority,Title,Text,Named Entities
0,23689,A21SYVGVNG8RAS,5,Low,Yummy snacks!,Popchips are the bomb!! I use the parmesan garlic to scoop up cottage cheese as a healthy alternative to chips and dip. My healthy eating program is saved.,[]
1,23690,AQJYXC0MPRQJL,5,Low,Great chip that is different from the rest,"I like the puffed nature of this chip that makes it more unique in the chip market. I ordered the Salt and Vinegar and absolutely love that flavor, hands down my favorite chip ever. I have tried the cheddar and regular flavors as well. The cheddar is about a 4/5 and the regular is about a 3/5 because I prefer strong flavors and obviously that would not be the case for the regular. The Salt and Vinegar is kind of weak compared to some regular S&V chips, but is quite flavorful and makes you wanting to come back for more.","[Salt and Vinegar, Salt and Vinegar, S & V]"
2,23691,A30NYUHEDLWI0Y,5,Low,Great Alternative to Potato Chips,"I just love these chips! I was always a big fan of potato chips, but haven't had one since I discovered popchips. They are great for dipping or all alone. I am constantly re-ordering them. One note however-if you are on a low salt diet these chips are probably not for you. They are high in sodium. We go through a case every two months. If you love them it pays to join the subscribe and save program through Amazon. You save money and stay supplied!",[Amazon]
3,23692,A2NU55U9LKTB5J,3,High,Not somthing I would crave,"These tasted like potatoe stix, that we got in grade school with our lunches usually on pizza day. They were the bomb then, not so much now. Won't buy again unless I get them for cheap or free.",[]
4,23693,A225F7QFP5LIW2,5,Low,healthy and delicious,"These chips are great! They look almost like a flattened rice cake, but taste so much better, more like a potato chip. The bbq flavor is delicious. They are very low in fat and full of flavor. It is easy to eat an entire bag of these!",[]


In [18]:
# Create a unique list of named entities

named_entities=list(set(df['Named Entities'].explode().dropna().tolist()))
named_entities[:20]

['Seattle',
 '##o Chip',
 '##os',
 'P',
 'Baked Lays',
 'com',
 'Black Pepper',
 'Vinegar',
 '##lestero',
 '##al',
 'BBQ Pop Chip',
 'Funyun',
 'Deal of the Day',
 'Cream & Onion',
 'Amazon Vine',
 'Jalapeno',
 'PopChips',
 '##rming',
 'CA',
 'Original Pop Chip']

### More Cleaned Named Entity

In [21]:
[entity for entity in named_entities if '#' not in entity][:20] # Excluding subwords

['Seattle',
 'P',
 'Baked Lays',
 'com',
 'Black Pepper',
 'Vinegar',
 'BBQ Pop Chip',
 'Funyun',
 'Deal of the Day',
 'Cream & Onion',
 'Amazon Vine',
 'Jalapeno',
 'PopChips',
 'CA',
 'Original Pop Chip',
 'So',
 'TX',
 'Po',
 'Pop',
 'Grandma Utz']